In [1]:
import pandas as pd
import uuid
import os

class ABTBuilder:
    """
    Pipeline automatizado para construção da Analytical Base Table (ABT) Agrícola.
    """
    def __init__(self, raw_metadata_path: str, output_path: str):
        self.raw_path = raw_metadata_path
        self.output_path = output_path
        self.df = None

    def load_and_merge_data(self):
        """Carrega os dados brutos e realiza joins com outras fontes (clima, talhão), se aplicável."""
        # Neste cenário inicial, a base primária são os metadados das imagens
        self.df = pd.read_csv(self.raw_path)
        
    def generate_unique_identifiers(self):
        """Garante a rastreabilidade criando um UUID v4 para cada amostra."""
        if 'sample_id' not in self.df.columns:
            self.df.insert(0, 'sample_id', [str(uuid.uuid4()) for _ in range(len(self.df))])

    def standardize_columns(self):
        """Aplica padronização Snake Case em todas as colunas para compatibilidade SQL/Python."""
        self.df.columns = (
            self.df.columns
            .str.strip()
            .str.lower()
            .str.replace(' ', '_')
            .str.replace('-', '_')
        )

    def save_abt(self):
        """Persiste a ABT no diretório de processados."""
        os.makedirs(os.path.dirname(self.output_path), exist_ok=True)
        self.df.to_csv(self.output_path, index=False)
        print(f"[OK] ABT construída com {self.df.shape[0]} registros e {self.df.shape[1]} colunas.")
        print(f"[OK] Salvo em: {self.output_path}")

# Execução do Pipeline
builder = ABTBuilder(
    raw_metadata_path='../data/processed/metadata_raw_images.csv', 
    output_path='../data/processed/abt_sanidade_vegetal.csv'
)
builder.load_and_merge_data()
builder.generate_unique_identifiers()
builder.standardize_columns()
builder.save_abt()

[OK] ABT construída com 6571 registros e 13 colunas.
[OK] Salvo em: ../data/processed/abt_sanidade_vegetal.csv


### 📖 Dicionário de Dados (Origem das Colunas)

| Coluna Padronizada | Origem do Dado | Descrição |
| :--- | :--- | :--- |
| `sample_id` | **Gerado via Pipeline (UUID)** | Identificador hash único global para a amostra. |
| `filepath` / `filename` | **Extrator do SO** | Caminho absoluto/relativo e nome do arquivo de imagem original. |
| `dataset_source` | **Roboflow / Mendeley** | Fonte primária de onde o lote de imagens foi adquirido. |
| `class_label` | **Anotação de Especialista** | Variável alvo contendo o diagnóstico (Saudável, Ferrugem, etc.). |
| `width` / `height` | **Pillow / OpenCV** | Dimensões estruturais da imagem em pixels. |
| `color_mode` / `channels` | **Pillow / OpenCV** | Espectro de cor capturado (ex: RGB = 3 canais). |
| `size_kb` | **Extrator do SO** | Peso físico do arquivo em disco, útil para identificar compressão excessiva. |